# Working with data 📊

Top-level `:=` variables are kept from cell to cell, `nb.Cache` computes an
expensive value once, and results can be shown as markdown, HTML or charts.

In [ ]:
import "encoding/json"

type Planet struct {
	Name     string  `json:"name"`
	DistAU   float64 `json:"dist_au"`
	RadiusKm float64 `json:"radius_km"`
	Moons    int     `json:"moons"`
	YearDays float64 `json:"year_days"`
}

const planetsJSON = `[
	{"name": "Mercury", "dist_au": 0.39, "radius_km": 2440, "moons": 0, "year_days": 88},
	{"name": "Venus", "dist_au": 0.72, "radius_km": 6052, "moons": 0, "year_days": 225},
	{"name": "Earth", "dist_au": 1.00, "radius_km": 6371, "moons": 1, "year_days": 365},
	{"name": "Mars", "dist_au": 1.52, "radius_km": 3390, "moons": 2, "year_days": 687},
	{"name": "Jupiter", "dist_au": 5.20, "radius_km": 69911, "moons": 95, "year_days": 4333},
	{"name": "Saturn", "dist_au": 9.58, "radius_km": 58232, "moons": 146, "year_days": 10759},
	{"name": "Uranus", "dist_au": 19.2, "radius_km": 25362, "moons": 28, "year_days": 30687},
	{"name": "Neptune", "dist_au": 30.1, "radius_km": 24622, "moons": 16, "year_days": 60190}
]`

func loadPlanets() []Planet {
	var ps []Planet
	if err := json.Unmarshal([]byte(planetsJSON), &ps); err != nil {
		panic(err)
	}
	return ps
}

`planets` is declared with `:=`, so its value is kept for the next cells:

In [ ]:
planets := loadPlanets()
len(planets)

In [ ]:
// planetTable formats planets as a markdown table.
func planetTable(ps []Planet) string {
	var sb strings.Builder
	sb.WriteString("| Planet | Distance (AU) | Radius (km) | Moons | Year (days) |\n")
	sb.WriteString("|---|--:|--:|--:|--:|\n")
	for _, p := range ps {
		fmt.Fprintf(&sb, "| **%s** | %.2f | %.0f | %d | %.0f |\n", p.Name, p.DistAU, p.RadiusKm, p.Moons, p.YearDays)
	}
	return sb.String()
}

nb.DisplayMarkdown(planetTable(planets))

Kepler's third law says a planet's year squared is proportional to its
distance cubed. Let's check, and sort the result:

In [ ]:
type kepler struct {
	Name  string
	Ratio float64 // (year in Earth years)² / (distance in AU)³: about 1
}

var ks []kepler
for _, p := range planets {
	years := p.YearDays / 365.25
	ks = append(ks, kepler{p.Name, years * years / (p.DistAU * p.DistAU * p.DistAU)})
}
slices.SortFunc(ks, func(a, b kepler) int { return cmp.Compare(a.Ratio, b.Ratio) })
for _, k := range ks {
	fmt.Printf("%-8s %.3f\n", k.Name, k.Ratio)
}

## Charts in plain text

In [ ]:
// barChart draws a horizontal bar chart with ANSI colors.
func barChart(labels []string, values []float64, width int) {
	maxV := slices.Max(values)
	for i, v := range values {
		n := int(v / maxV * float64(width))
		fmt.Printf("%-8s \x1b[38;5;%dm%s\x1b[0m %g\n", labels[i], 39+i*6, strings.Repeat("█", n)+"▏", v)
	}
}

var names []string
var moons []float64
for _, p := range planets {
	names = append(names, p.Name)
	moons = append(moons, float64(p.Moons))
}
barChart(names, moons, 50)

## HTML output

In [ ]:
// HTML output is drawn as text here, and saved as HTML for Jupyter.
gonbui.DisplayHTML(`
<h3>The gas giants</h3>
<ul>
	<li><b>Jupiter</b> is bigger than all the other planets combined.</li>
	<li><b>Saturn</b> is less dense than <i>water</i>.</li>
</ul>
<p>Two more, <code>Uranus</code> and <code>Neptune</code>, are ice giants.</p>
`)

## Caching

Every cell runs in a new process, so a slow `var` initializer would run again
in every later cell. `nb.Cache` stores the result in the kernel's workspace:
the first run takes a while, later ones are instant. `%cache` lists the
stored values and `%cache clear` forgets them.

In [ ]:
// countPrimes counts the primes below n with a sieve of Eratosthenes.
func countPrimes(n int) int {
	composite := make([]bool, n)
	count := 0
	for i := 2; i < n; i++ {
		if composite[i] {
			continue
		}
		count++
		for j := i * i; j < n; j += i {
			composite[j] = true
		}
	}
	return count
}

var primes = nb.Cache("primes below 50M", func() int {
	fmt.Println("counting primes below 50 million (only once)…")
	return countPrimes(50_000_000)
})

primes

In [ ]:
// No counting this time: the cached value is read back.
fmt.Printf("%d primes, %.2f%% of the numbers below 50 million\n", primes, float64(primes)/50e4)

In [ ]:
%cache